# ContractBuilder Node — Unit Tests
Step-by-step testing of the ContractBuilder Node.
Run each cell independently to inspect the generated ServiceContract and logs.

The ContractBuilder makes **2 real LLM calls** (locator + extractor). No RAG, no Qdrant needed.

In [1]:
# Step 1 — Imports and setup
# Run this cell first before any other cell.

from openapi_rulesbank.config import get_logger
from openapi_rulesbank.config.paths import TSPEC_DATA_TEST_FILE
from openapi_rulesbank.nodes.reader import reader_node
from openapi_rulesbank.nodes.contract_builder import contract_builder_node, _build_sections_summary
from openapi_rulesbank.schemas.contract import ServiceContract, format_contract_summary

# The service name the contract is built for. It is the service *name*, not a YAML.
TARGET_SERVICE = 'ProvMnS'

logger = get_logger(__name__)
logger.info('Imports OK.')

/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-06 00:26:01 [INFO] __main__: Imports OK.


Using Device: cuda


In [2]:
# Step 2 — Run Reader Node to prepare state
# The ContractBuilder depends on parsed_sections, so we run the Reader first.
# This is the same state that LangGraph would pass between the two nodes.

reader_state = {
    'main_doc_path':       TSPEC_DATA_TEST_FILE,
    'auxiliary_doc_paths': [],
    'target_yaml':         TARGET_SERVICE,
}

reader_result = reader_node(reader_state)

assert len(reader_result['parsed_sections']) > 0, 'Reader returned no sections'

logger.info(f"Reader OK — {len(reader_result['parsed_sections'])} section(s) ready for ContractBuilder.")

2026-07-06 00:26:01 [INFO] openapi_rulesbank.nodes.reader: Reader Node started.
2026-07-06 00:26:01 [INFO] openapi_rulesbank.nodes.reader: Reader sections: 726 total = 725 for planner + 1 excluded by reader
2026-07-06 00:26:01 [DEBUG] openapi_rulesbank.nodes.reader: Loaded 0 auxiliary document(s).
2026-07-06 00:26:01 [DEBUG] openapi_rulesbank.tools.document_tools: Loaded OpenAPI reference v3.0.1 from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/openapi_rulesbank/config/../../../data/references/openapi_reference/openapi_spec_3.0.1.md'.
2026-07-06 00:26:01 [INFO] openapi_rulesbank.nodes.reader: Reader Node complete.
2026-07-06 00:26:01 [INFO] __main__: Reader OK — 725 section(s) ready for ContractBuilder.


In [3]:
# Step 3 — Test _build_sections_summary()
# Verifies the helper formats sections into the locator table: section_id | title | preview

summary = _build_sections_summary(reader_result['parsed_sections'])

assert isinstance(summary, str) and len(summary) > 0, 'Expected non-empty summary string'

lines = summary.splitlines()
assert all('|' in l for l in lines[:5]), 'Expected id | title | preview format in each line'

logger.info(f'_build_sections_summary OK — {len(lines)} lines generated.')
logger.info('Preview (first 3 lines):')
for line in lines[:3]:
    logger.info(f'  {line[:120]}')

2026-07-06 00:26:01 [INFO] __main__: _build_sections_summary OK — 725 lines generated.
2026-07-06 00:26:01 [INFO] __main__: Preview (first 3 lines):
2026-07-06 00:26:01 [INFO] __main__:   1 | 3.1 Definitions | For the purposes of the present document, the terms and definitions given in TR 21.905 \[1\] and t
2026-07-06 00:26:01 [INFO] __main__:   2 | 3.2 Abbreviations | For the purposes of the present document, the abbreviations given in TR 21.905 \[1\] and the fol
2026-07-06 00:26:01 [INFO] __main__:   3 | 11.1 Generic provisioning management service | #


In [4]:
# Step 4 — Run contract_builder_node() end-to-end
# Makes 2 real LLM API calls (locator + extractor). Ensure OPENAI_API_KEY is set in .env.

contract_state = {
    'parsed_sections': reader_result['parsed_sections'],
    'target_yaml':     TARGET_SERVICE,
}

contract_result = contract_builder_node(contract_state)

assert 'service_contract' in contract_result
contract = contract_result['service_contract']

# Round-trips through the schema — raises if any field has the wrong shape/type.
ServiceContract(**contract)
assert contract['service_name'] == TARGET_SERVICE

oos = contract['out_of_scope_signals']

# The contract must scope the target on two axes:
#  (1) stage 3 block identified (so the Planner extracts only from the solution set),
#  (2) out-of-scope signals contain BOTH the other MnS (named services) AND at
#      least one out-of-scope section block (pattern "<clause>.x (out of scope)").
assert contract['stage_scope'].strip(), 'stage_scope is empty — locator did not find the stage 3 block'
assert any('(out of scope)' in x and 'service' in x.lower() for x in oos), \
    'No other Management Service flagged as out of scope'
assert any(x.split(' ')[0].endswith('.x') for x in oos), \
    'No out-of-scope section block (e.g. "11.1.x (out of scope)") recorded'

logger.info('contract_builder_node OK')
logger.info(f"  capabilities        : {len(contract['capabilities'])}")
logger.info(f"  key_concepts        : {len(contract['key_concepts'])}")
logger.info(f"  inclusion_keywords  : {len(contract['inclusion_keywords'])}")
logger.info(f"  exclusion_keywords  : {len(contract['exclusion_keywords'])}")
logger.info(f"  out_of_scope        : {len(oos)} signal(s)")
logger.info(f"  stage_scope set     : {bool(contract['stage_scope'].strip())}")

2026-07-06 00:26:01 [INFO] openapi_rulesbank.nodes.contract_builder: ContractBuilder Node started.
2026-07-06 00:26:05 [INFO] openapi_rulesbank.nodes.contract_builder: ContractBuilder locator: purpose=['4'], capabilities=['5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54'], concepts=['52', '53', '54']
2026-07-06 00:26:14 [INFO] openapi_rulesbank.nodes.contract_builder: ContractBuilder complete: 9 capabilities, 19 inclusion keyword(s), 0 exclusion keyword(s).
2026-07-06 00:26:14 [INFO] __main__: contract_builder_node OK
2026-07-06 00:26:14 [INFO] __main__:   capabilities        : 9
2026-07-06 00:26:14 [INFO] __main__:   key_concepts        : 25
2026-07-06 00:26:14 [INFO] __main__:   inclusion_keywords  : 19
2026-07-06 00:26:14 [INFO] __main__:   exclusion_keyw

In [5]:
# Step 5 — Inspect the generated contract (formatted summary)
# This is the exact block the Planner and Extractor receive as scope context.

print(format_contract_summary(contract))

SERVICE CONTRACT
Service: ProvMnS
Purpose: This service provides generic provisioning services for managing managed objects, including Create, Read, Update, and Delete (CRUD) operations and notifications related to managed objects and their attributes.
Capabilities: createMOI, deleteMOI, modifyMOIAttributes, getMOIAttributes, notifyMOICreation, notifyMOIDeletion, notifyMOIAttributeValueChanges, notifyEvent, notifyMOIChanges
Key concepts: Managed Object Instance, ManagedEntity DN, ManagedEntity class, attribute name, attribute value, modificationListOut, status, notificationId, notificationType, eventTime, systemDN, correlatedNotifications, sourceIndicator, attributeListIn, attributeListOut, baseObjectInstance, scopeType, scopeLevel, filter, modificationList, moiChanges, op, path, value, oldValue
Inclusion keywords: Managed Object Instance, MOI, ManagedEntity, attributeListIn, attributeListOut, modificationList, notificationId, notificationType, notifyMOICreation, notifyMOIDeletion, not

In [6]:
# Step 6 — Inspect capabilities and scope in detail
# Each capability: name, type (operation|notification|state|subscription|other), description.

for i, cap in enumerate(contract['capabilities']):
    print(f"[{i}] ({cap['type']}) {cap['name']}")
    print(f"     {cap['description']}")

print(f"\nKey concepts       : {contract['key_concepts']}")
print(f"Inclusion keywords : {contract['inclusion_keywords']}")
print(f"Exclusion keywords : {contract['exclusion_keywords']}")
print(f"Out-of-scope       : {contract['out_of_scope_signals']}")

# Stage 2 vs stage 3 scope — tells the Planner/Extractor to extract ONLY from
# the target's stage 3 block (the RESTful HTTP solution set).
print(f"\nStage scope        :\n  {contract['stage_scope']}")

[0] (operation) createMOI
     This operation is invoked by MnS consumer to request the MnS producer to create a Managed Object instance in the MIB maintained by MnS producer.
[1] (operation) deleteMOI
     This operation is invoked by MnS consumer to request the deletion of one or more Managed Object instances in the MIB maintained by the MnS producer.
[2] (operation) modifyMOIAttributes
     This operation is invoked by MnS consumer to request the modification of one or more Managed Object instances from MnS producer.
[3] (operation) getMOIAttributes
     This operation is invoked by MnS consumer to request the retrieval of management information (Managed Object attribute names and values) from the MIB maintained by MnS producer.
[4] (notification) notifyMOICreation
     This notification notifies the subscribed consumers that a new Managed Object Instance has been created.
[5] (notification) notifyMOIDeletion
     This notification notifies the subscribed consumers that an existing 

In [7]:
# Step 7 — Save the full contract JSON for inspection
# Written to data/outputs/contracts/ so the generated contract can be reviewed later.

import json
import os
from datetime import datetime
from openapi_rulesbank.config.paths import OUTPUTS_RULES_DIR

# OUTPUTS_RULES_DIR is .../data/outputs/rules_bank — go up one to data/outputs/.
contracts_dir = os.path.join(os.path.dirname(OUTPUTS_RULES_DIR), 'contracts')
os.makedirs(contracts_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = os.path.join(contracts_dir, f'contract_{TARGET_SERVICE}_{timestamp}.json')

with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(contract, f, indent=2, ensure_ascii=False)

logger.info(f'Contract saved to: {out_path}')

2026-07-06 00:26:14 [INFO] __main__: Contract saved to: /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/openapi_rulesbank/config/../../../data/outputs/contracts/contract_ProvMnS_20260706_002614.json
